# Prajna AI: `sft` v5
Init = prajna-ctx4k (4096 ctx, India continued pretraining). Data: full smol-smoltalk + sarvamai/samvaad-hi-v1 (Latin-script only) + 1500 Prajna identity convs, packed at 4096. Eval: v3 / v4 / v5 on the same 1024-packed val sets as v4, plus v5 on its 4096-packed val.

In [ ]:
import os, time, glob
NB_START = time.time()
os.makedirs('/tmp/kaggle-llm/src/llm', exist_ok=True)
os.makedirs('/tmp/kaggle-llm/scripts', exist_ok=True)
%cd /tmp/kaggle-llm
!pip install -q tiktoken 2>&1 | tail -1

In [ ]:
%%writefile src/llm/__init__.py
from .config import GPTConfig, PRESETS, get_config
from .model import GPT

__all__ = ["GPTConfig", "PRESETS", "get_config", "GPT"]


In [ ]:
%%writefile src/llm/config.py
"""Model architecture configs and size presets."""
from dataclasses import asdict, dataclass, replace


@dataclass
class GPTConfig:
    vocab_size: int = 50304      # GPT-2 BPE (50257) padded to a multiple of 64 for speed
    block_size: int = 1024       # context length
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    ffn_hidden: int = 2048       # SwiGLU hidden size (~8/3 * d_model)
    rope_theta: float = 10000.0
    dropout: float = 0.0


PRESETS = {
    # smoke tests only
    "tiny": GPTConfig(n_layer=4, n_head=4, d_model=128, ffn_hidden=384, block_size=256),
    # ~34M: trains in ~2h on Kaggle
    "30m": GPTConfig(n_layer=8, n_head=8, d_model=384, ffn_hidden=1024, block_size=512),
    # ~124M (GPT-2 small size): the best fit for Kaggle's free GPUs
    "124m": GPTConfig(),
}


def get_config(name: str) -> GPTConfig:
    return replace(PRESETS[name])


def to_dict(cfg: GPTConfig) -> dict:
    return asdict(cfg)


In [ ]:
%%writefile src/llm/model.py
"""LLaMA-style decoder-only Transformer: RoPE + RMSNorm + SwiGLU, no biases, tied embeddings."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import GPTConfig


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()  # normalize in fp32 for fp16 stability
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


def rope_cache(seq_len: int, head_dim: int, theta: float):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    freqs = torch.outer(torch.arange(seq_len, dtype=torch.float32), inv_freq)  # (T, head_dim/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    # x: (B, H, T, D); rotate the two halves of the head dimension
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class Attention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.proj.is_residual = True
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))


class SwiGLU(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.w13 = nn.Linear(cfg.d_model, 2 * cfg.ffn_hidden, bias=False)  # gate + up, fused
        self.w2 = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)
        self.w2.is_residual = True

    def forward(self, x):
        gate, up = self.w13(x).chunk(2, dim=-1)
        return self.w2(F.silu(gate) * up)


class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        return x + self.mlp(self.norm2(x))


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

        cos, sin = rope_cache(cfg.block_size, cfg.d_model // cfg.n_head, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02
            if getattr(m, "is_residual", False):
                std /= math.sqrt(2 * self.cfg.n_layer)
            nn.init.normal_(m.weight, mean=0.0, std=std)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok_emb.weight.numel() if non_embedding else n

    def forward(self, idx, targets=None):
        T = idx.size(1)
        assert T <= self.cfg.block_size, f"sequence length {T} > block_size {self.cfg.block_size}"
        x = self.tok_emb(idx)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        for blk in self.blocks:
            x = blk(x, cos, sin)
        x = self.norm_f(x)
        if targets is None:
            return self.lm_head(x[:, [-1], :]), None  # only the last position is needed
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

    def configure_optimizer(self, weight_decay, lr, device_type):
        params = [p for p in self.parameters() if p.requires_grad]
        groups = [
            {"params": [p for p in params if p.dim() >= 2], "weight_decay": weight_decay},
            {"params": [p for p in params if p.dim() < 2], "weight_decay": 0.0},
        ]
        return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95), eps=1e-8,
                                 fused=(device_type == "cuda"))

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50, eos_id=None, repetition_penalty=1.0):
        """repetition_penalty > 1 discourages tokens already produced in this generation (CTRL-style)."""
        start = idx.size(1)
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            logits = logits[:, -1, :].float()
            if repetition_penalty != 1.0 and idx.size(1) > start:
                prev = idx[:, start:]
                seen = logits.gather(1, prev)
                logits.scatter_(1, prev, torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty))
            logits = logits / max(temperature, 1e-5)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
            idx = torch.cat([idx, nxt], dim=1)
            if eos_id is not None and (nxt == eos_id).all():
                break
        return idx


In [ ]:
%%writefile src/llm/data.py
"""Token data loaders reading uint16 .bin files produced by scripts/prepare_data.py."""
import os

import numpy as np
import torch


class TokenLoader:
    """Walks a shuffled list of non-overlapping chunks, sharded across DDP ranks.

    Deterministic given (seed, rank, world), so a resumed run can skip exactly the
    micro-batches it already consumed and continue with fresh data.

    SFT data (prepare_sft.py) is packed in blocks of block_size+1 with a uint8 `<name>_mask.bin`
    next to it: targets whose mask is 0 are set to -1 (ignored by the loss).
    """

    def __init__(self, path, block_size, batch_size, device, rank=0, world=1, seed=1337, skip_batches=0):
        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        mask_path = path[:-len(".bin")] + "_mask.bin"
        self.mask = np.memmap(mask_path, dtype=np.uint8, mode="r") if os.path.exists(mask_path) else None
        self.T, self.B, self.device = block_size, batch_size, device
        self.stride = block_size + 1 if self.mask is not None else block_size
        self.rank, self.world, self.seed = rank, world, seed
        self.n_chunks = (len(self.data) - block_size - 1) // self.stride + 1
        assert self.n_chunks >= world * batch_size, f"{path} is too small for this batch size"
        self.per_epoch = (self.n_chunks // world) // batch_size  # micro-batches per epoch per rank
        self._set_position(skip_batches)

    def _set_position(self, batch_idx):
        self.epoch, self.batch_in_epoch = divmod(batch_idx, self.per_epoch)
        self._make_order()

    def _make_order(self):
        perm = np.random.default_rng(self.seed + self.epoch).permutation(self.n_chunks)
        self.order = perm[self.rank::self.world]

    def reset(self):
        self._set_position(0)

    def next(self):
        if self.batch_in_epoch >= self.per_epoch:
            self.epoch, self.batch_in_epoch = self.epoch + 1, 0
            self._make_order()
        s = self.batch_in_epoch * self.B
        starts = self.order[s:s + self.B] * self.stride
        self.batch_in_epoch += 1
        buf = np.stack([self.data[i:i + self.T + 1] for i in starts]).astype(np.int64)
        buf = torch.from_numpy(buf)
        x, y = buf[:, :-1], buf[:, 1:].clone()
        if self.mask is not None:
            m = np.stack([self.mask[i + 1:i + self.T + 1] for i in starts])
            y[torch.from_numpy(m) == 0] = -1
        if "cuda" in str(self.device):
            return x.pin_memory().to(self.device, non_blocking=True), y.pin_memory().to(self.device, non_blocking=True)
        return x.to(self.device), y.to(self.device)


class SyntheticLoader:
    """Random tokens, for throughput benchmarks without a dataset."""

    def __init__(self, vocab_size, block_size, batch_size, device):
        self.V, self.T, self.B, self.device = vocab_size, block_size, batch_size, device

    def reset(self):
        pass

    def next(self):
        x = torch.randint(0, self.V, (self.B, self.T + 1), device=self.device)
        return x[:, :-1], x[:, 1:]


In [ ]:
%%writefile src/llm/chat.py
"""Chat template shared by SFT data prep and generation.

    <|system|>\n...\n<|user|>\n...\n<|assistant|>\n...<|endoftext|>\n<|user|>\n...

Role tags are plain text (GPT-2 BPE has no special tokens for them); each assistant turn
ends with <|endoftext|> so generation stops there. Only assistant content + EOS is trained on.
"""
EOS = 50256
ROLE_TAG = {"system": "<|system|>\n", "user": "<|user|>\n", "assistant": "<|assistant|>\n"}


def encode_turns(enc, messages):
    """-> list of (ids, target_mask) per message; mask=1 marks tokens the model learns to produce."""
    out = []
    for m in messages:
        role, text = m["role"], m["content"].strip()
        if role == "assistant":
            head = enc.encode_ordinary(ROLE_TAG["assistant"])
            body = enc.encode_ordinary(text) + [EOS]
            tail = enc.encode_ordinary("\n")
            out.append((head + body + tail, [0] * len(head) + [1] * len(body) + [0] * len(tail)))
        else:
            ids = enc.encode_ordinary(ROLE_TAG.get(role, ROLE_TAG["user"]) + text + "\n")
            out.append((ids, [0] * len(ids)))
    return out


def prompt_ids(enc, messages):
    """Encode a conversation so far, ending with the assistant tag, ready for generation."""
    ids = []
    for t, _ in encode_turns(enc, messages):
        ids += t
    return ids + enc.encode_ordinary(ROLE_TAG["assistant"])


In [ ]:
%%writefile src/llm/identity.py
"""Synthetic "who are you?" conversations mixed into SFT so the model describes itself correctly.

smol-smoltalk has no consistent identity (assistants call themselves "an AI assistant designed to ...",
or play roles from system prompts), so the chat model otherwise invents a name and creator.
These are templated from FACTS; edit FACTS to change what the model says about itself.

    from llm.identity import identity_conversations
    convs = identity_conversations(n=600, seed=0)   # list of [{"role", "content"}, ...]
"""
import random

FACTS = {
    "name": "Prajna AI",
    "creator": "somdevsheel",
    "params": "124 million",
    "arch": "LLaMA-style decoder-only transformer",
    "pretrain": "about 1.9 billion tokens of educational web text (FineWeb-Edu)",
    "sft": "general chat conversations (smol-smoltalk) and India-focused conversations in English and Hinglish",
    "hardware": "free Kaggle GPUs (two NVIDIA T4s)",
    "context": "1,024 tokens",
}

NAME_Q = ["Who are you?", "What is your name?", "What's your name?", "who r u", "Introduce yourself.",
          "Can you tell me about yourself?", "What are you?", "Are you a person or a bot?",
          "Hi! Who am I talking to?", "Tell me a bit about who you are.", "What should I call you?",
          "Are you an AI?", "hey, what are you exactly?", "Please introduce yourself briefly."]
CREATOR_Q = ["Who made you?", "Who created you?", "Who trained you?", "Who built you?", "Who is your developer?",
             "Which company made you?", "Who are your creators?", "who programmed you"]
OTHER_Q = ["Are you ChatGPT?", "Are you Claude?", "Are you made by OpenAI?", "Are you Gemini from Google?",
           "Are you GPT-4?", "Is this Llama from Meta?", "Are you Siri?"]
TECH_Q = ["How many parameters do you have?", "How big is your model?", "What architecture are you based on?",
          "How were you trained?", "What data were you trained on?", "What hardware were you trained on?",
          "How long a text can you read at once?", "Tell me the technical details of your model."]
LIMIT_Q = ["Can you browse the internet?", "Do you remember our previous conversations?",
           "Are your answers always correct?", "Can I trust everything you say?",
           "What are your limitations?", "Do you know today's news?"]
HI_NAME_Q = ["Tum kaun ho?", "Aap kaun ho?", "Tumhara naam kya hai?", "Aapka naam kya hai?", "Apne baare mein batao."]
HI_CREATOR_Q = ["Tumhe kisne banaya?", "Aapko kisne banaya hai?", "Tumhe kisne train kiya?"]
HI_OTHER_Q = ["Kya tum ChatGPT ho?", "Kya aap Google ka Gemini ho?", "Kya tum OpenAI ke ho?"]
HI_LANG_Q = ["Kya tum Hindi samajhte ho?", "Kya main Hinglish mein baat kar sakta hoon?", "Tum kya kar sakte ho?"]
COMPARE_Q = ["Are you as smart as ChatGPT?", "Are you better than GPT-4?", "Are you as good as the big AI models?"]
GREETINGS = ["Hello!", "Hi", "hey there", "Good morning!", "Hi, how are you?", "Hello, nice to meet you."]
SYSTEMS = ["You are a helpful assistant.", "You are a friendly and helpful AI assistant.",
           "You are a helpful assistant. Answer concisely."]


def _answers(f):
    intro = [
        f"I'm {f['name']}, a small language model with {f['params']} parameters. I was trained from scratch by "
        f"{f['creator']} on {f['hardware']}. I can answer questions, explain things and help with writing, "
        f"but I'm small, so please double-check important facts.",
        f"My name is {f['name']}. I'm an AI assistant: a {f['params']}-parameter {f['arch']} built by "
        f"{f['creator']}. How can I help you today?",
        f"I'm an AI language model called {f['name']}. {f['creator']} trained me from scratch on {f['hardware']}. "
        f"I'm happy to help with questions, explanations or writing.",
        f"I'm {f['name']}, a compact AI assistant created by {f['creator']}. I'm not a person, just a "
        f"{f['params']}-parameter language model. What would you like to talk about?",
    ]
    creator = [
        f"I was created by {f['creator']}, who trained me from scratch on {f['hardware']}. I'm {f['name']}, "
        f"a {f['params']}-parameter language model.",
        f"{f['creator']} built and trained me. I'm a small, independent project, not a product of a large AI "
        f"company.",
        f"I was trained by {f['creator']}. First I learned general language from {f['pretrain']}, then I was "
        f"fine-tuned on {f['sft']} to become a chat assistant.",
    ]
    other = [
        f"No. I'm {f['name']}, a much smaller model ({f['params']} parameters) trained from scratch by "
        f"{f['creator']}. I'm not affiliated with that model or the company behind it.",
        f"No, I'm not. I'm {f['name']}, a small independent language model built by {f['creator']} on "
        f"{f['hardware']}. I'm far smaller than the big commercial assistants, so I make more mistakes.",
    ]
    tech = [
        f"I'm a {f['arch']} with {f['params']} parameters and a context window of {f['context']}. "
        f"I was pretrained on {f['pretrain']} and then fine-tuned on {f['sft']}, all on {f['hardware']}.",
        f"Here are my details:\n\n- Name: {f['name']}\n- Size: {f['params']} parameters\n"
        f"- Architecture: {f['arch']}\n- Pretraining: {f['pretrain']}\n- Chat fine-tuning: {f['sft']}\n"
        f"- Hardware: {f['hardware']}\n- Context window: {f['context']}\n- Built by: {f['creator']}",
        f"I have {f['params']} parameters, which is small for a language model. I was trained in two stages: "
        f"pretraining on {f['pretrain']}, then chat fine-tuning on {f['sft']}.",
    ]
    limits = [
        "I can't browse the internet, and I don't remember past conversations; I only see the current chat. "
        "My knowledge comes from my training data, and because I'm a small model I often get facts wrong, "
        "so please verify anything important.",
        f"I'm a small model ({f['params']} parameters), so I make mistakes and sometimes state things that "
        "sound confident but are wrong. I have no internet access and no knowledge of recent events. "
        "I'm best for simple explanations, brainstorming and practice, not for facts you need to rely on.",
    ]
    compare = [
        f"No. I'm much smaller than assistants like ChatGPT, so my answers are less reliable. I can't access the "
        "internet or remember earlier chats. Treat what I say as a starting point and double-check it.",
    ]
    hi_name = [
        f"Main {f['name']} hoon, ek chhota AI assistant. Mujhe {f['creator']} ne Kaggle ke free GPUs par scratch se "
        f"train kiya hai. Aap mujhse sawaal pooch sakte ho, main madad karne ki koshish karunga.",
        f"Mera naam {f['name']} hai. Main {f['params']} parameters wala ek language model hoon, jise "
        f"{f['creator']} ne banaya hai. Bataiye, main aapki kya madad kar sakta hoon?",
    ]
    hi_creator = [
        f"Mujhe {f['creator']} ne banaya hai. Main {f['name']} hoon, kisi badi company ka product nahi, ek chhota "
        f"independent project hoon. Isliye kabhi kabhi galti kar sakta hoon, important facts zaroor check karein.",
        f"{f['creator']} ne mujhe Kaggle ke free GPUs par scratch se train kiya hai. Main {f['name']} hoon.",
    ]
    hi_other = [
        f"Nahi, main woh nahi hoon. Main {f['name']} hoon, {f['creator']} ka banaya hua ek chhota model. "
        f"Main bade AI models se kaafi chhota hoon, isliye zyada galtiyan kar sakta hoon.",
    ]
    hi_lang = [
        f"Haan, main Hinglish samajh leta hoon aur usi mein jawab de sakta hoon. Main {f['name']} hoon. "
        f"Aap English ya Hinglish, jismein comfortable ho, usmein poochiye.",
        f"Main sawaalon ke jawab de sakta hoon, cheezein simple tarike se samjha sakta hoon, aur writing mein madad "
        f"kar sakta hoon, English ya Hinglish mein. Bas yaad rakhiye ki main chhota model hoon, facts check kar lijiye.",
    ]
    greet = [
        f"Hello! I'm {f['name']}, a small AI assistant. What can I help you with today?",
        f"Hi there! I'm {f['name']}. Ask me a question or tell me what you're working on.",
    ]
    return {"name": (NAME_Q, intro), "creator": (CREATOR_Q, creator), "other": (OTHER_Q, other),
            "tech": (TECH_Q, tech), "limits": (LIMIT_Q, limits),
            "compare": (COMPARE_Q, compare), "hi_name": (HI_NAME_Q, hi_name), "hi_creator": (HI_CREATOR_Q, hi_creator),
            "hi_other": (HI_OTHER_Q, hi_other), "hi_lang": (HI_LANG_Q, hi_lang)}, greet


def identity_conversations(n=600, seed=0, facts=None):
    """n conversations: 1-3 identity Q&A turns, sometimes after a greeting or a generic system prompt."""
    f = dict(FACTS, **(facts or {}))
    cats, greet = _answers(f)
    keys = list(cats)
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        msgs = []
        if rng.random() < 0.25:
            msgs.append({"role": "system", "content": rng.choice(SYSTEMS)})
        if rng.random() < 0.2:
            msgs += [{"role": "user", "content": rng.choice(GREETINGS)},
                     {"role": "assistant", "content": rng.choice(greet)}]
        for k in rng.sample(keys, rng.choice([1, 1, 1, 2, 2, 3])):
            qs, ans = cats[k]
            q = rng.choice(qs)
            if rng.random() < 0.15:
                q = q.lower().rstrip("?.!")
            msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": rng.choice(ans)}]
        out.append(msgs)
    return out


In [ ]:
%%writefile scripts/train.py
"""Train the GPT on 1 or more GPUs (DDP), with fp16/bf16 AMP, grad accumulation,
warmup + cosine LR, periodic checkpoints, a session time limit, and exact resume.

Single GPU:   python scripts/train.py --preset 124m --data_dir data/fineweb
Multi GPU:    torchrun --standalone --nproc_per_node=gpu scripts/train.py --preset 124m ...
Benchmark:    torchrun --standalone --nproc_per_node=gpu scripts/train.py --synthetic --max_iters 20
Resume:       ... --resume path/to/ckpt.pt   (or --resume auto  -> <out_dir>/ckpt.pt if present)
Fine-tune:    ... --init_from base/model.pt --data_dir data/sft   (SFT data from prepare_sft.py)
"""
import argparse
import json
import math
import os
import sys
import time
from contextlib import nullcontext

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from llm.config import PRESETS, GPTConfig, get_config, to_dict  # noqa: E402
from llm.chat import prompt_ids  # noqa: E402
from llm.data import SyntheticLoader, TokenLoader  # noqa: E402
from llm.model import GPT  # noqa: E402

SAMPLE_PROMPTS = ["The most important thing to understand about science is",
                  "Once upon a time,"]
CHAT_PROMPTS = ["What is photosynthesis? Explain it simply.",
                "Write a short poem about the ocean."]


# settings a resumed run takes from its checkpoint: LR schedule, batch geometry, data order
RESUME_KEEP = ["lr", "min_lr_ratio", "warmup_iters", "weight_decay", "grad_clip", "batch_tokens",
               "micro_batch", "data_offset_iters", "seed", "eval_every", "ckpt_every"]


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument("--preset", default="124m", choices=list(PRESETS))
    p.add_argument("--data_dir", default="data/fineweb")
    p.add_argument("--synthetic", action="store_true", help="random tokens; throughput benchmark")
    p.add_argument("--out_dir", default="checkpoints/run")
    p.add_argument("--resume", default="", help="ckpt path, or 'auto' for <out_dir>/ckpt.pt")
    p.add_argument("--init_from", default="", help="start from these weights (model.pt) with a fresh "
                   "optimizer/schedule, e.g. for fine-tuning; architecture comes from the file")
    p.add_argument("--block_size", type=int, default=0,
                   help="override context length (e.g. 4096 with --init_from to extend a 1024 model)")
    p.add_argument("--rope_theta", type=float, default=0, help="override RoPE base (raise it when extending context)")
    p.add_argument("--init_optimizer", action="store_true",
                   help="with --init_from a full ckpt.pt: also load its AdamW moments (smoother continued pretraining)")
    p.add_argument("--data_offset_iters", type=int, default=0,
                   help="skip this many optimizer steps of data first (continue on unseen data; same micro_batch/GPUs)")
    p.add_argument("--total_tokens", type=float, default=2e9, help="training budget (sets the LR schedule)")
    p.add_argument("--max_iters", type=int, default=0, help="override total_tokens")
    p.add_argument("--batch_tokens", type=int, default=2 ** 19, help="global tokens per optimizer step")
    p.add_argument("--micro_batch", type=int, default=8, help="sequences per GPU per forward pass")
    p.add_argument("--lr", type=float, default=1.2e-3)
    p.add_argument("--min_lr_ratio", type=float, default=0.1)
    p.add_argument("--warmup_iters", type=int, default=200)
    p.add_argument("--weight_decay", type=float, default=0.1)
    p.add_argument("--grad_clip", type=float, default=1.0)
    p.add_argument("--eval_every", type=int, default=250)
    p.add_argument("--eval_batches", type=int, default=20)
    p.add_argument("--ckpt_every", type=int, default=250)
    p.add_argument("--log_every", type=int, default=10)
    p.add_argument("--time_limit_hours", type=float, default=0, help="checkpoint & exit after this long")
    p.add_argument("--time_budget_hours", type=float, default=0,
                   help="fresh runs: measure speed, then shrink the schedule so it finishes in this time")
    p.add_argument("--compile", action="store_true")
    p.add_argument("--amp", default="auto", choices=["auto", "fp16", "bf16", "fp32"])
    p.add_argument("--seed", type=int, default=1337)
    return p.parse_args()


def lr_at(it, a, max_iters):
    if it < a.warmup_iters:
        return a.lr * (it + 1) / a.warmup_iters
    progress = min(1.0, (it - a.warmup_iters) / max(1, max_iters - a.warmup_iters))
    min_lr = a.lr * a.min_lr_ratio
    return min_lr + 0.5 * (1 + math.cos(math.pi * progress)) * (a.lr - min_lr)


def main():
    a = get_args()

    # ---------------- distributed setup ----------------
    ddp = "RANK" in os.environ
    if ddp:
        dist.init_process_group("nccl")
        rank, world, local_rank = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
        device = f"cuda:{local_rank}"
        torch.cuda.set_device(device)
    else:
        rank, world, local_rank = 0, 1, 0
        device = "cuda" if torch.cuda.is_available() else "cpu"
    master = rank == 0
    device_type = "cuda" if "cuda" in device else "cpu"

    def log(*msg):
        if master:
            print(*msg, flush=True)

    torch.manual_seed(a.seed + rank)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # bf16 on Ampere+, fp16 + loss scaling on T4/P100
    if device_type == "cuda":
        if a.amp == "auto":
            amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
        else:
            amp_dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[a.amp]
        autocast = torch.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype != torch.float32)
    else:
        amp_dtype, autocast = torch.float32, nullcontext()
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

    # ---------------- resume ----------------
    ckpt_path = os.path.join(a.out_dir, "ckpt.pt") if a.resume == "auto" else a.resume
    ckpt = None
    if ckpt_path and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        cfg = GPTConfig(**ckpt["model_config"])
        log(f"Resuming from {ckpt_path} at iter {ckpt['iter']}")
        # the schedule and the data order must match the interrupted run, whatever flags were passed now
        saved = ckpt.get("args", {})
        for k in RESUME_KEEP:
            if k in saved and getattr(a, k) != saved[k]:
                log(f"  resume: {k} = {saved[k]} (from checkpoint; ignoring {getattr(a, k)})")
                setattr(a, k, saved[k])
    else:
        if a.resume and a.resume != "auto":
            raise FileNotFoundError(ckpt_path)
        init = None
        if a.init_from:
            init = torch.load(a.init_from, map_location="cpu", weights_only=False)
            cfg = GPTConfig(**init["model_config"])
            log(f"Initializing weights from {a.init_from}")
        else:
            cfg = get_config(a.preset)
        if a.block_size or a.rope_theta:
            from dataclasses import replace
            cfg = replace(cfg, **({"block_size": a.block_size} if a.block_size else {}),
                          **({"rope_theta": a.rope_theta} if a.rope_theta else {}))
            log(f"Config override: block_size={cfg.block_size}, rope_theta={cfg.rope_theta:g}")

    # ---------------- batch geometry ----------------
    tokens_per_micro = a.micro_batch * cfg.block_size * world
    accum = max(1, a.batch_tokens // tokens_per_micro)
    step_tokens = accum * tokens_per_micro
    if ckpt:
        max_iters = ckpt["max_iters"]
        assert ckpt["step_tokens"] == step_tokens, "batch geometry changed; use the same micro_batch/GPUs"
    else:
        max_iters = a.max_iters or math.ceil(a.total_tokens / step_tokens)
    start_iter = ckpt["iter"] if ckpt else 0

    # ---------------- model / optimizer ----------------
    model = GPT(cfg).to(device)
    optimizer = model.configure_optimizer(a.weight_decay, a.lr, device_type)
    if ckpt:
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        if ckpt.get("scaler"):
            scaler.load_state_dict(ckpt["scaler"])
    elif a.init_from:
        model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in init["model"].items()})
        if a.init_optimizer:
            assert "optimizer" in init, "--init_optimizer needs a full ckpt.pt, not a weights-only model.pt"
            optimizer.load_state_dict(init["optimizer"])
            if init.get("scaler"):
                scaler.load_state_dict(init["scaler"])
            log("Loaded optimizer state from the init checkpoint")
        del init
    best_val = ckpt.get("best_val", float("inf")) if ckpt else float("inf")
    resumed = ckpt is not None
    del ckpt

    raw_model = model
    if a.compile:
        model = torch.compile(model)
    if ddp:
        model = DDP(model, device_ids=[local_rank])

    log(f"GPUs: {world} x {torch.cuda.get_device_name(0) if device_type == 'cuda' else 'cpu'} | AMP {amp_dtype}")
    log(f"Model: {raw_model.num_params() / 1e6:.1f}M params "
        f"({raw_model.num_params(non_embedding=True) / 1e6:.1f}M non-embedding) | {to_dict(cfg)}")
    log(f"Batch: {step_tokens:,} tokens/step = {a.micro_batch} seq x {cfg.block_size} x {world} GPU x {accum} accum")
    log(f"Schedule: {max_iters:,} iters = {max_iters * step_tokens / 1e9:.2f}B tokens (from iter {start_iter})")

    # ---------------- data ----------------
    if a.synthetic:
        train_loader = SyntheticLoader(cfg.vocab_size, cfg.block_size, a.micro_batch, device)
        val_loader = None
    else:
        train_loader = TokenLoader(os.path.join(a.data_dir, "train.bin"), cfg.block_size, a.micro_batch,
                                   device, rank, world, a.seed,
                                   skip_batches=(start_iter + a.data_offset_iters) * accum)
        val_loader = TokenLoader(os.path.join(a.data_dir, "val.bin"), cfg.block_size, a.micro_batch,
                                 device, rank, world, a.seed)
        if a.data_offset_iters:
            left = (train_loader.per_epoch - train_loader.batch_in_epoch) * a.micro_batch * cfg.block_size * world
            log(f"Data offset: skipping {a.data_offset_iters} steps already trained on; "
                f"{left / 1e9:.2f}B unseen tokens left in this epoch")
        log(f"Data: {len(train_loader.data) / 1e9:.2f}B train tokens, {len(val_loader.data) / 1e6:.1f}M val tokens"
            + (" | SFT (loss on assistant tokens only)" if train_loader.mask is not None else ""))
    sft = not a.synthetic and train_loader.mask is not None

    tokenizer = None
    if master and not a.synthetic:
        import tiktoken
        tokenizer = tiktoken.get_encoding("gpt2")

    @torch.no_grad()
    def evaluate():
        model.eval()
        val_loader.reset()
        total = torch.zeros(1, device=device)
        for _ in range(a.eval_batches):
            x, y = val_loader.next()
            with autocast:
                _, loss = model(x, y)
            total += loss.float()
        total /= a.eval_batches
        if ddp:
            dist.all_reduce(total, op=dist.ReduceOp.AVG)
        model.train()
        return total.item()

    @torch.no_grad()
    def sample():
        raw_model.eval()
        for prompt in (CHAT_PROMPTS if sft else SAMPLE_PROMPTS):
            ids = prompt_ids(tokenizer, [{"role": "user", "content": prompt}]) if sft else tokenizer.encode(prompt)
            idx = torch.tensor([ids], device=device)
            with autocast:
                out = raw_model.generate(idx, 100 if sft else 60, temperature=0.8, top_k=50, eos_id=50256)
            log(f"  > {tokenizer.decode(out[0].tolist())!r}")
        raw_model.train()

    def save(it, path_name="ckpt.pt"):
        if not master or a.synthetic:
            return
        os.makedirs(a.out_dir, exist_ok=True)
        tmp = os.path.join(a.out_dir, path_name + ".tmp")
        torch.save({"model": raw_model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "model_config": to_dict(cfg), "iter": it,
                    "max_iters": max_iters, "step_tokens": step_tokens, "best_val": best_val,
                    "args": vars(a)}, tmp)
        os.replace(tmp, os.path.join(a.out_dir, path_name))

    def export_weights():
        """Small fp16 weights-only file for inference."""
        if not master or a.synthetic:
            return
        sd = {k: v.half() if v.is_floating_point() else v for k, v in raw_model.state_dict().items()}
        torch.save({"model": sd, "model_config": to_dict(cfg)}, os.path.join(a.out_dir, "model.pt"))

    log_file = None
    if master and not a.synthetic:
        os.makedirs(a.out_dir, exist_ok=True)
        log_file = open(os.path.join(a.out_dir, "log.jsonl"), "a")
    session_start = time.time()
    it = start_iter
    step_times = []
    BUDGET_PROBE = 20  # iterations to time before fitting the schedule to the budget

    # ---------------- training loop ----------------
    while it < max_iters:
        last = it == max_iters - 1

        if val_loader is not None and a.eval_every and (it % a.eval_every == 0 or last):
            val_loss = evaluate()
            best_val = min(best_val, val_loss)
            log(f"[eval] iter {it} | val loss {val_loss:.4f} | ppl {math.exp(val_loss):.1f}")
            if log_file:
                log_file.write(json.dumps({"iter": it, "val_loss": val_loss}) + "\n")
                log_file.flush()
            if master:
                sample()

        lr = lr_at(it, a, max_iters)
        for g in optimizer.param_groups:
            g["lr"] = lr

        t0 = time.time()
        loss_accum = torch.zeros(1, device=device)
        for micro in range(accum):
            x, y = train_loader.next()
            sync = model.no_sync() if (ddp and micro < accum - 1) else nullcontext()
            with sync:
                with autocast:
                    _, loss = model(x, y)
                    loss = loss / accum
                scaler.scale(loss).backward()
            loss_accum += loss.detach().float()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), a.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        if device_type == "cuda":
            torch.cuda.synchronize()
        if ddp:
            dist.all_reduce(loss_accum, op=dist.ReduceOp.AVG)
        dt = time.time() - t0
        it += 1

        # fit the LR schedule to the time budget (fresh runs only, decided once, during warmup)
        if a.time_budget_hours and not resumed and it - start_iter <= BUDGET_PROBE:
            if it - start_iter > 2:  # skip the first steps (CUDA / compile warmup)
                step_times.append(dt)
            if it - start_iter == BUDGET_PROBE:
                per_iter = sum(step_times) / len(step_times) * 1.05  # +5% for eval/ckpt overhead
                remaining = a.time_budget_hours * 3600 - (time.time() - session_start)
                new_max = torch.tensor([min(max_iters, it + int(remaining / per_iter))], device=device)
                if ddp:
                    dist.broadcast(new_max, src=0)
                old, max_iters = max_iters, int(new_max.item())
                log(f"[budget] {per_iter:.2f}s/iter -> schedule {old:,} -> {max_iters:,} iters "
                    f"= {max_iters * step_tokens / 1e9:.2f}B tokens in {a.time_budget_hours:.1f}h")

        if it % a.log_every == 0 or it == start_iter + 1:
            elapsed = time.time() - session_start
            tps = step_tokens / dt
            eta_h = (max_iters - it) * dt / 3600
            log(f"iter {it:6d}/{max_iters} | loss {loss_accum.item():.4f} | lr {lr:.2e} | "
                f"norm {grad_norm.item():.2f} | {dt * 1000:.0f}ms | {tps:,.0f} tok/s | "
                f"elapsed {elapsed / 60:.1f}m | eta {eta_h:.1f}h")
            if log_file:
                log_file.write(json.dumps({"iter": it, "loss": loss_accum.item(), "lr": lr,
                                           "grad_norm": grad_norm.item(), "tok_per_s": tps}) + "\n")
                log_file.flush()

        # time limit: all ranks agree before stopping
        stop = torch.tensor([0.0], device=device)
        if a.time_limit_hours and (time.time() - session_start) > a.time_limit_hours * 3600:
            stop += 1
        if ddp:
            dist.all_reduce(stop, op=dist.ReduceOp.MAX)

        if a.ckpt_every and (it % a.ckpt_every == 0) or it == max_iters or stop.item():
            save(it)
            export_weights()
            log(f"[ckpt] saved at iter {it}")
        if stop.item():
            log(f"Time limit reached at iter {it}/{max_iters}. Resume with --resume <out_dir>/ckpt.pt")
            break

    if it >= max_iters:
        log(f"Training complete. Best val loss {best_val:.4f}")
    if log_file:
        log_file.close()
    if ddp:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/prepare_sft.py
"""Build packed SFT data from a chat dataset (default: HuggingFaceTB/smol-smoltalk).

Writes to --out_dir:
  train.bin / val.bin            uint16 tokens, whole conversations packed into blocks of block_size+1
  train_mask.bin / val_mask.bin  uint8, 1 where the token is an assistant target
  meta.json

Conversations longer than a block are cut at a turn boundary (must keep >= 1 assistant turn).
--identity N adds N synthetic "who are you?" conversations (src/llm/identity.py) to train only,
so val stays the first --val_rows conversations of the dataset and val loss is comparable across runs.
--mix_repo adds a second chat dataset (e.g. sarvamai/samvaad-hi-v1) to train; with --mix_latin_only,
conversations containing Devanagari (poorly tokenized by GPT-2 BPE) are skipped. Its first --mix_val_rows
kept conversations go to val_mix.bin instead, a held-out set for that domain (scripts/eval_sft.py).

    python scripts/prepare_sft.py --out_dir data/sft --max_rows 200000 --identity 1000
"""
import argparse
import glob
import re
import json
import os
import sys
import time

import numpy as np
import tiktoken

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from llm.chat import EOS, encode_turns  # noqa: E402
from llm.identity import identity_conversations  # noqa: E402


DEVANAGARI = re.compile(r"[\u0900-\u097F]")


def iter_rows(a, repo=None, parquet="", shards=None):
    import pyarrow.parquet as pq
    repo, shards = repo or a.repo, shards or a.shards
    if parquet:
        files = sorted(glob.glob(parquet))
    else:
        from huggingface_hub import HfApi, hf_hub_download
        names = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                       if f.startswith("data/train") and f.endswith(".parquet"))
        files = [hf_hub_download(repo, f, repo_type="dataset", local_dir=a.cache_dir) for f in names[:shards]]
    print(f"{len(files)} parquet files", flush=True)
    for f in files:
        for batch in pq.ParquetFile(f).iter_batches(batch_size=4096, columns=["messages"]):
            yield from batch.column(0).to_pylist()


def fit(turns, limit):
    """Keep the longest prefix of turns that fits in `limit` and ends with an assistant turn."""
    ids, mask, best = [], [], None
    for t, m in turns:
        if len(ids) + len(t) > limit:
            break
        ids, mask = ids + t, mask + m
        if any(m):
            best = (list(ids), list(mask))
    return best


class Packer:
    """First-fit packing of whole conversations into fixed blocks (pool of open blocks)."""

    def __init__(self, block, prefix, pool=256):
        self.block, self.pool = block, pool
        self.tok = open(prefix + ".bin", "wb")
        self.msk = open(prefix + "_mask.bin", "wb")
        self.open = []  # list of [ids, mask]
        self.blocks = self.targets = self.used = 0

    def add(self, ids, mask):
        for b in self.open:
            if len(b[0]) + len(ids) <= self.block:
                b[0] += ids
                b[1] += mask
                break
        else:
            self.open.append([list(ids), list(mask)])
        if len(self.open) > self.pool:  # retire the fullest block
            self._write(self.open.pop(max(range(len(self.open)), key=lambda i: len(self.open[i][0]))))

    def _write(self, b):
        ids, mask = b
        pad = self.block - len(ids)
        self.tok.write(np.array(ids + [EOS] * pad, dtype=np.uint16).tobytes())
        self.msk.write(np.array(mask + [0] * pad, dtype=np.uint8).tobytes())
        self.targets += sum(mask)
        self.used += len(ids)
        self.blocks += 1

    def close(self):
        for b in self.open:
            self._write(b)
        self.open = []
        self.tok.close()
        self.msk.close()


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--repo", default="HuggingFaceTB/smol-smoltalk")
    p.add_argument("--parquet", default="", help="glob of local parquet files instead of downloading")
    p.add_argument("--shards", type=int, default=2, help="train parquet shards to download")
    p.add_argument("--max_rows", type=int, default=0)
    p.add_argument("--val_rows", type=int, default=2000)
    p.add_argument("--identity", type=int, default=0, help="synthetic identity conversations added to train")
    p.add_argument("--mix_repo", default="", help="second chat dataset added to train (same messages format)")
    p.add_argument("--mix_parquet", default="", help="glob of local parquet files for the mix dataset")
    p.add_argument("--mix_shards", type=int, default=8)
    p.add_argument("--mix_max_rows", type=int, default=0)
    p.add_argument("--mix_val_rows", type=int, default=0, help="held out to val_mix.bin")
    p.add_argument("--mix_latin_only", action="store_true", help="skip conversations containing Devanagari")
    p.add_argument("--block_size", type=int, default=1024)
    p.add_argument("--out_dir", default="data/sft")
    p.add_argument("--cache_dir", default="data/hf_cache")
    a = p.parse_args()

    os.makedirs(a.out_dir, exist_ok=True)
    enc = tiktoken.get_encoding("gpt2")
    block = a.block_size + 1
    val = Packer(block, os.path.join(a.out_dir, "val"))
    train = Packer(block, os.path.join(a.out_dir, "train"))
    t0, n, kept, cut = time.time(), 0, 0, 0
    for msgs in iter_rows(a, parquet=a.parquet):
        if a.max_rows and n >= a.max_rows:
            break
        n += 1
        turns = encode_turns(enc, msgs)
        full = sum(len(t) for t, _ in turns)
        ex = fit(turns, block)
        if ex is None:
            continue
        cut += len(ex[0]) < full
        kept += 1
        (val if kept <= a.val_rows else train).add(*ex)
        if n % 20000 == 0:
            print(f"{n:,} rows | {train.blocks:,} train blocks | {time.time() - t0:.0f}s", flush=True)
    mix = {"read": 0, "kept": 0, "skipped_script": 0, "truncated": 0}
    if a.mix_repo or a.mix_parquet:
        val_mix = Packer(block, os.path.join(a.out_dir, "val_mix")) if a.mix_val_rows else None
        for msgs in iter_rows(a, repo=a.mix_repo, parquet=a.mix_parquet, shards=a.mix_shards):
            if a.mix_max_rows and mix["read"] >= a.mix_max_rows:
                break
            mix["read"] += 1
            if a.mix_latin_only and any(DEVANAGARI.search(m["content"]) for m in msgs):
                mix["skipped_script"] += 1
                continue
            turns = encode_turns(enc, msgs)
            ex = fit(turns, block)
            if ex is None:
                continue
            mix["truncated"] += len(ex[0]) < sum(len(t) for t, _ in turns)
            mix["kept"] += 1
            (val_mix if val_mix and mix["kept"] <= a.mix_val_rows else train).add(*ex)
        if val_mix:
            val_mix.close()
            mix["val_blocks"], mix["val_target_tokens"] = val_mix.blocks, val_mix.targets
        print(f"mix {a.mix_repo or a.mix_parquet}: {mix}", flush=True)
    ident = 0
    for msgs in identity_conversations(a.identity):
        ex = fit(encode_turns(enc, msgs), block)
        if ex is not None:
            train.add(*ex)
            ident += 1
    val.close()
    train.close()
    meta = {"repo": a.repo, "rows_read": n, "rows_kept": kept, "rows_truncated": cut, "identity_rows": ident,
            "block_size": a.block_size, "train_blocks": train.blocks, "val_blocks": val.blocks,
            "train_tokens": train.blocks * block, "fill": round(train.used / max(1, train.blocks * block), 3),
            "train_target_tokens": train.targets,
            "val_target_tokens": val.targets, "mix_repo": a.mix_repo or a.mix_parquet, "mix": mix}
    json.dump(meta, open(os.path.join(a.out_dir, "meta.json"), "w"), indent=1)
    print(f"Done in {time.time() - t0:.0f}s: {meta}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/eval_sft.py
"""Mean assistant-token loss of a model on a packed SFT file (e.g. val_mix.bin from prepare_sft.py).

    python scripts/eval_sft.py --ckpt checkpoints/sft-v3/model.pt --data data/sft/val_mix.bin
"""
import argparse
import math
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from llm.config import GPTConfig  # noqa: E402
from llm.model import GPT  # noqa: E402


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", nargs="+", required=True)
    p.add_argument("--data", required=True)
    p.add_argument("--batch", type=int, default=8)
    p.add_argument("--block", type=int, default=0,
                   help="packed block size of --data (default: the model's block_size); lets a 4096-context "
                        "model be scored on 1024-packed data")
    a = p.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    data = np.memmap(a.data, dtype=np.uint16, mode="r")
    mask = np.memmap(a.data[:-len(".bin")] + "_mask.bin", dtype=np.uint8, mode="r")
    for path in a.ckpt:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        cfg = GPTConfig(**ck["model_config"])
        model = GPT(cfg)
        model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in ck["model"].items()})
        model.to(dev).eval()
        S = (a.block or cfg.block_size) + 1
        assert S - 1 <= cfg.block_size, f"--block {S - 1} > model block_size {cfg.block_size}"
        n = len(data) // S
        tot, cnt = 0.0, 0
        with torch.no_grad(), torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):
            for i in range(0, n, a.batch):
                buf = torch.from_numpy(np.array(data[i * S:min(n, i + a.batch) * S], dtype=np.int64)).view(-1, S)
                m = torch.from_numpy(np.array(mask[i * S:min(n, i + a.batch) * S])).view(-1, S)[:, 1:]
                x, y = buf[:, :-1].to(dev), buf[:, 1:].clone()
                y[m == 0] = -1
                logits = model.lm_head(model.norm_f(_body(model, x)))
                loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.to(dev).view(-1),
                                       ignore_index=-1, reduction="sum")
                tot += loss.item()
                cnt += int((y != -1).sum())
        print(f"{path}: {n} blocks, {cnt:,} target tokens, loss {tot / cnt:.4f} (ppl {math.exp(tot / cnt):.2f})",
              flush=True)


def _body(model, x):
    T = x.size(1)
    h = model.tok_emb(x)
    cos, sin = model.rope_cos[:T], model.rope_sin[:T]
    for blk in model.blocks:
        h = blk(h, cos, sin)
    return h


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/generate.py
"""Generate text from a trained model.

    python scripts/generate.py --ckpt checkpoints/run/model.pt --prompt "The history of the Roman Empire"
    python scripts/generate.py --ckpt checkpoints/run/model.pt --interactive
    python scripts/generate.py --ckpt checkpoints/sft/model.pt --chat              # multi-turn chat
    python scripts/generate.py --ckpt checkpoints/sft/model.pt --chat --prompt "What is DNA?"
"""
import argparse
import os
import sys

import tiktoken
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from llm.chat import prompt_ids  # noqa: E402
from llm.config import GPTConfig  # noqa: E402
from llm.model import GPT  # noqa: E402


def load(path, device):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = GPT(GPTConfig(**ckpt["model_config"]))
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in ckpt["model"].items()})
    return model.to(device).eval()


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", default="checkpoints/run/model.pt")
    p.add_argument("--prompt", default="The most important thing to understand about science is")
    p.add_argument("--tokens", type=int, default=200)
    p.add_argument("--temperature", type=float, default=None, help="default 0.8 (0.7 with --chat)")
    p.add_argument("--top_k", type=int, default=50)
    p.add_argument("--num_samples", type=int, default=1)
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--interactive", action="store_true")
    p.add_argument("--chat", action="store_true", help="fine-tuned model: chat template (+ multi-turn REPL)")
    p.add_argument("--system", default="", help="--chat: optional system prompt")
    p.add_argument("--rep_penalty", type=float, default=None, help="repetition penalty (default 1.0, 1.15 with --chat)")
    a = p.parse_args()
    if a.temperature is None:
        a.temperature = 0.7 if a.chat else 0.8
    if a.rep_penalty is None:
        a.rep_penalty = 1.15 if a.chat else 1.0

    if a.seed is not None:
        torch.manual_seed(a.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = load(a.ckpt, device)
    enc = tiktoken.get_encoding("gpt2")
    print(f"Loaded {model.num_params() / 1e6:.1f}M params on {device}")

    def run(prompt):
        idx = torch.tensor([enc.encode(prompt)], device=device)
        for _ in range(a.num_samples):
            with torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
                out = model.generate(idx, a.tokens, a.temperature, a.top_k, eos_id=50256,
                                     repetition_penalty=a.rep_penalty)
            print(enc.decode(out[0].tolist()).replace("<|endoftext|>", ""))
            print("-" * 60)

    def reply(messages):
        idx = torch.tensor([prompt_ids(enc, messages)[-model.cfg.block_size + a.tokens:]], device=device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
            out = model.generate(idx, a.tokens, a.temperature, a.top_k, eos_id=50256,
                                     repetition_penalty=a.rep_penalty)
        return enc.decode(out[0, idx.size(1):].tolist()).replace("<|endoftext|>", "").strip()

    if a.chat:
        base = [{"role": "system", "content": a.system}] if a.system else []
        if a.prompt and not a.interactive and "--prompt" in sys.argv:
            for _ in range(a.num_samples):
                print(reply(base + [{"role": "user", "content": a.prompt}]))
                print("-" * 60)
            return
        history = list(base)
        print("Chat mode. Empty line or Ctrl-D quits, /reset clears history.")
        while True:
            try:
                msg = input("you> ").strip()
            except (EOFError, KeyboardInterrupt):
                break
            if not msg:
                break
            if msg == "/reset":
                history = list(base)
                continue
            history.append({"role": "user", "content": msg})
            ans = reply(history)
            history.append({"role": "assistant", "content": ans})
            print(f"bot> {ans}\n")
        return

    if a.interactive:
        while True:
            try:
                prompt = input("prompt> ").strip()
            except (EOFError, KeyboardInterrupt):
                break
            if prompt:
                run(prompt)
    else:
        run(a.prompt)


if __name__ == "__main__":
    main()


In [ ]:
!mkdir -p /kaggle/working/code && cp -r src scripts /kaggle/working/code/

In [ ]:
def find(pattern):
    hits = sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True))
    print(pattern, '->', hits)
    return hits
BASE = [h for h in find('ctx4k/model.pt')][0]
sft = find('sft/model.pt')
V4 = [h for h in sft if 'prajna' in h][0]
V3 = [h for h in sft if 'kaggle-llm' in h][0]
os.environ.update(BASE=BASE, V3=V3, V4=V4)
import torch
for p in (BASE, V3, V4):
    print(p, torch.load(p, map_location='cpu', weights_only=False)['model_config'])

In [ ]:
!python scripts/prepare_sft.py --shards 4 --identity 1500 --block_size 4096 --mix_repo sarvamai/samvaad-hi-v1 --mix_latin_only --mix_val_rows 600 --out_dir /tmp/sft --cache_dir /tmp/hf_cache

In [ ]:
# same val sets as v4 (first 2000 smoltalk / 600 samvaad kept rows, 1024-packed)
!python scripts/prepare_sft.py --shards 1 --max_rows 2100 --block_size 1024 --mix_repo sarvamai/samvaad-hi-v1 --mix_latin_only --mix_max_rows 3500 --mix_val_rows 600 --out_dir /tmp/sft1k --cache_dir /tmp/hf_cache

In [ ]:
import json
meta = json.load(open('/tmp/sft/meta.json'))
print(json.dumps(meta, indent=1))
TOTAL = meta['train_tokens']  # one epoch over the mix
elapsed_h = (time.time() - NB_START) / 3600
BUDGET = round(7.0 - elapsed_h - 0.5, 2)
print(f'train tokens {TOTAL / 1e6:.1f}M (~{TOTAL // 131072} iters) | prep {elapsed_h * 60:.0f} min -> budget {BUDGET} h')

In [ ]:
!torchrun --standalone --nproc_per_node=gpu scripts/train.py --init_from $BASE --data_dir /tmp/sft --out_dir /kaggle/working/sft --time_budget_hours {BUDGET} --time_limit_hours {BUDGET + 0.2} --micro_batch 2 --batch_tokens 131072 --total_tokens {TOTAL} --lr 2e-4 --warmup_iters 50 --eval_every 100 --eval_batches 10 --ckpt_every 200 --log_every 10
!cp /tmp/sft/meta.json /kaggle/working/sft/data_meta.json

In [ ]:
!echo '== smol-smoltalk val, 1024-packed (v3 / v4 / v5)' | tee /kaggle/working/eval.txt
!python scripts/eval_sft.py --ckpt $V3 $V4 /kaggle/working/sft/model.pt --data /tmp/sft1k/val.bin --block 1024 | tee -a /kaggle/working/eval.txt
!echo '== samvaad held-out, 1024-packed (v3 / v4 / v5)' | tee -a /kaggle/working/eval.txt
!python scripts/eval_sft.py --ckpt $V3 $V4 /kaggle/working/sft/model.pt --data /tmp/sft1k/val_mix.bin --block 1024 | tee -a /kaggle/working/eval.txt
!echo '== v5 on its own 4096-packed val sets' | tee -a /kaggle/working/eval.txt
!python scripts/eval_sft.py --ckpt /kaggle/working/sft/model.pt --data /tmp/sft/val.bin --block 4096 | tee -a /kaggle/working/eval.txt
!python scripts/eval_sft.py --ckpt /kaggle/working/sft/model.pt --data /tmp/sft/val_mix.bin --block 4096 | tee -a /kaggle/working/eval.txt

In [ ]:
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "Who are you and who made you?" | tee -a /kaggle/working/samples.txt
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "Tum kaun ho aur tumhe kisne banaya?" | tee -a /kaggle/working/samples.txt
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "What is the capital of Kerala and what is it known for?" | tee -a /kaggle/working/samples.txt
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "Tell me about the festival of Diwali." | tee -a /kaggle/working/samples.txt
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "Explain the role of the Lok Sabha in Indian democracy." | tee -a /kaggle/working/samples.txt
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "Mujhe Taj Mahal ke baare mein batao." | tee -a /kaggle/working/samples.txt
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "What is photosynthesis? Explain it simply." | tee -a /kaggle/working/samples.txt
!python scripts/generate.py --ckpt /kaggle/working/sft/model.pt --chat --num_samples 2 --seed 0 --tokens 250 --prompt "Write a Python function that checks whether a number is prime." | tee -a /kaggle/working/samples.txt